## Dataframe.transform()
The `transform()` method in PySpark allows you to apply a custom transformation function to a DataFrame. This is useful for chaining multiple operations or reusing transformation logic.

**Example:**
python
def add_column(df):
    return df.withColumn("new_col", df["existing_col"] + 1)

df_transformed = df.transform(add_column)

In [0]:
data = [(1,"ravi",5000),(2,"raj",6000),(3,"ram",7000),(4,"raj",8000),(5,"ram",9000)]
schema = ["id", "name", "salary"]
df = spark.createDataFrame(data,schema)
df.show()


+---+----+------+
| id|name|salary|
+---+----+------+
|  1|ravi|  5000|
|  2| raj|  6000|
|  3| ram|  7000|
|  4| raj|  8000|
|  5| ram|  9000|
+---+----+------+



In [0]:
from pyspark.sql.functions import *
def changenametoUpper(df):
    df = df.withColumn("name",upper(col("name")))
    return df
def doublesalary(df):
    df = df.withColumn("salary",col("salary")*2)
    return df
df1 = df.transform(changenametoUpper)
df1.show()
df2 = df1.transform(doublesalary)
df2.show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  1|RAVI|  5000|
|  2| RAJ|  6000|
|  3| RAM|  7000|
|  4| RAJ|  8000|
|  5| RAM|  9000|
+---+----+------+

+---+----+------+
| id|name|salary|
+---+----+------+
|  1|RAVI| 10000|
|  2| RAJ| 12000|
|  3| RAM| 14000|
|  4| RAJ| 16000|
|  5| RAM| 18000|
+---+----+------+



## functions.transform()
The `functions.transform()` function in PySpark is used to apply a transformation to each element in an array column. It returns a new array with the transformed values.

**Example:**
python
from pyspark.sql.functions import transform

### Assume df has a column 'numbers' which is an array of integers
df = df.withColumn("doubled_numbers", transform(col("numbers"), lambda x: x * 2))


In [0]:
data = [(1,'kanth',['python','streamlit']),(2,'raju',['sql','pyspark']),(3,'sai',['sql','java'])]
schema = ['id','name','skills']
df = spark.createDataFrame(data,schema)
df.show()
df.printSchema()

+---+-----+-------------------+
| id| name|             skills|
+---+-----+-------------------+
|  1|kanth|[python, streamlit]|
|  2| raju|     [sql, pyspark]|
|  3|  sai|        [sql, java]|
+---+-----+-------------------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
def addnewskill(df):
    df = df.withColumn("skills", array_union(col("skills"), array(lit('scala'))))
    return df
df1 = df.transform(addnewskill)
display(df1)
df2 = df1.withColumn("skills",transform("skills",lambda x : upper(x)).alias("skills"))
df2.show()

id,name,skills
1,kanth,"List(python, streamlit, scala)"
2,raju,"List(sql, pyspark, scala)"
3,sai,"List(sql, java, scala)"


+---+-----+--------------------+
| id| name|              skills|
+---+-----+--------------------+
|  1|kanth|[PYTHON, STREAMLI...|
|  2| raju|[SQL, PYSPARK, SC...|
|  3|  sai|  [SQL, JAVA, SCALA]|
+---+-----+--------------------+



## createOrReplaceTempView()
The `createOrReplaceTempView()` method in PySpark allows you to register a DataFrame as a temporary table (view) in the Spark SQL catalog. This enables you to run SQL queries against the DataFrame using `spark.sql()`.

**Example:**
python
df.createOrReplaceTempView("employee_view")
result = spark.sql("SELECT * FROM employee_view WHERE salary > 6000")
result.show()

In [0]:
data = [(1,'raj','Male',2000),(2,'venky','Male',3000),(3,'srinidhi','Female',4000)]
schema = ['id','name','gender','salary']
df = spark.createDataFrame(data, schema)
df.show()
df.createOrReplaceTempView("employees")
df1 = spark.sql("""
                select id,name
                from employees
                """)
df1.show()

+---+--------+------+------+
| id|    name|gender|salary|
+---+--------+------+------+
|  1|     raj|  Male|  2000|
|  2|   venky|  Male|  3000|
|  3|srinidhi|Female|  4000|
+---+--------+------+------+

+---+--------+
| id|    name|
+---+--------+
|  1|     raj|
|  2|   venky|
|  3|srinidhi|
+---+--------+



In [0]:
%sql
select name,salary
from employees

name,salary
raj,2000
venky,3000
srinidhi,4000


In [0]:
spark.catalog.dropTempView("employees")

True

## createOrReplaceGlobalTempView()
The `createOrReplaceGlobalTempView()` method in PySpark registers a DataFrame as a global temporary view, accessible across all Spark sessions using the `global_temp` database.

**Example:**
python
df.createOrReplaceGlobalTempView("employee_global_view")
result = spark.sql("SELECT * FROM global_temp.employee_global_view WHERE salary > 3000")
result.show()